# Who has what: access examples

Answers "which roles does a user hold", "what can a user finally do", "who holds a role", "who reaches an object"
from `SNOWFLAKE.ACCOUNT_USAGE`. Read-only: no business data, nothing written except the optional functions of path A.

**Needs:** read access to `SNOWFLAKE.ACCOUNT_USAGE` (`IMPORTED PRIVILEGES` on database `SNOWFLAKE`, or the database
role `SNOWFLAKE.SECURITY_VIEWER`) and a warehouse. The data lags up to about 2 hours.

**Choose one path:**
- **A. Create the table functions first** (needs `CREATE FUNCTION` on a schema), then call them in one line.
- **B. Run the queries directly**: nothing is created; each cell holds the full query.

Run the inputs cell, then either the cells of A or those of B. Both give the same rows.

In [ ]:
-- Replace with yours. Used by every example below.
SET USER_NAME = 'ANALYST_01';
SET ROLE_NAME = 'F_ANALYST';
SET OBJECT_NAME = 'SALES.CORE.TBL_CUSTOMER';   -- DB, DB.SCHEMA or DB.SCHEMA.OBJECT
SET PRIVILEGE = 'SELECT';                       -- '' = any
SET MAX_DEPTH = 50;
SET MIN_DEPTH = 2;                              -- example 7: more than this many roles
SET SHOW_BUILTIN = FALSE;                       -- path B only

## Path A: create the functions

Replace `AQ_DB.AQ_SCHEMA` with a database and schema of yours in every cell below (find and replace), then run the five
`CREATE` cells. If one fails (a recursive query or ACCOUNT_USAGE not allowed in a function in your account), use path B.
A function reads ACCOUNT_USAGE with its owner's rights: grant `USAGE` on it only to roles that may see every grant.

In [ ]:
CREATE OR REPLACE FUNCTION AQ_DB.AQ_SCHEMA.AQ_USER_ROLES(USER_NAME VARCHAR, MAX_DEPTH NUMBER DEFAULT 50, SHOW_BUILTIN BOOLEAN DEFAULT FALSE)
RETURNS TABLE (ROLE_NAME VARCHAR, ROLE_KIND VARCHAR, DEPTH NUMBER, PATH VARCHAR, WAYS NUMBER)
AS
$$
WITH RECURSIVE
ARGS AS (SELECT CAST(USER_NAME AS VARCHAR) AS A_USER,
                CAST(COALESCE(MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST(SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD,
           CASE WHEN G.GRANTED_ON = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS CHILD_KIND
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
ME AS (   -- joins, not IN: a NULL name would empty NOT IN
    SELECT (SELECT COUNT(*) FROM ALL_USERS X JOIN ARGS A ON X.USR = A.A_USER) AS IS_USER,
           (SELECT COUNT(*) FROM UR U JOIN ARGS A ON U.USR = A.A_USER WHERE U.ROLE_NAME = 'PUBLIC') AS HAS_PUBLIC),
STARTS AS (   -- the user's granted roles, plus PUBLIC when it is not granted
    SELECT U.ROLE_NAME AS NODE FROM UR U JOIN ARGS A ON U.USR = A.A_USER
    UNION ALL
    SELECT 'PUBLIC' FROM ME M WHERE M.IS_USER > 0 AND M.HAS_PUBLIC = 0),
WALK (NODE, KIND, DEPTH, PATH, SEEN) AS (   -- every simple path down; SEEN = the roles on it, so a cycle stops
    SELECT CAST(S.NODE AS VARCHAR), CAST('ROLE' AS VARCHAR), CAST(1 AS INTEGER),
           CAST(A.A_USER || ' > ' || S.NODE AS VARCHAR), CAST(CHR(31) || S.NODE || CHR(31) AS VARCHAR)
    FROM STARTS S CROSS JOIN ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.CHILD, E.CHILD_KIND, W.DEPTH + 1, W.PATH || ' > ' || E.CHILD, W.SEEN || E.CHILD || CHR(31)
    FROM WALK W JOIN EDGE E ON E.PARENT = W.NODE CROSS JOIN ARGS A
    WHERE W.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.CHILD || CHR(31) IN W.SEEN) = 0)
SELECT W.NODE AS ROLE_NAME, W.KIND AS ROLE_KIND, W.DEPTH, W.PATH,
       COUNT(*) OVER (PARTITION BY W.NODE) AS WAYS
FROM WALK W
QUALIFY ROW_NUMBER() OVER (PARTITION BY W.NODE ORDER BY W.DEPTH, W.PATH) = 1
$$;

In [ ]:
CREATE OR REPLACE FUNCTION AQ_DB.AQ_SCHEMA.AQ_USER_ACCESS(USER_NAME VARCHAR, MAX_DEPTH NUMBER DEFAULT 50, SHOW_BUILTIN BOOLEAN DEFAULT FALSE)
RETURNS TABLE (PRIVILEGE VARCHAR, GRANTED_ON VARCHAR, OBJECT_NAME VARCHAR, VIA_ROLE VARCHAR, DEPTH NUMBER,
               PATH VARCHAR, WAYS NUMBER)
AS
$$
WITH RECURSIVE
ARGS AS (SELECT CAST(USER_NAME AS VARCHAR) AS A_USER,
                CAST(COALESCE(MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST(SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
PRIV AS (   -- every other grant is a privilege on an object (OWNERSHIP of a role too)
    SELECT G.GRANTEE, G.PRIVILEGE, G.GRANTED_ON, G.OBJ
    FROM GR G
    WHERE NOT (G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
ME AS (   -- joins, not IN: a NULL name would empty NOT IN
    SELECT (SELECT COUNT(*) FROM ALL_USERS X JOIN ARGS A ON X.USR = A.A_USER) AS IS_USER,
           (SELECT COUNT(*) FROM UR U JOIN ARGS A ON U.USR = A.A_USER WHERE U.ROLE_NAME = 'PUBLIC') AS HAS_PUBLIC),
STARTS AS (   -- the user's granted roles, plus PUBLIC when it is not granted
    SELECT U.ROLE_NAME AS NODE FROM UR U JOIN ARGS A ON U.USR = A.A_USER
    UNION ALL
    SELECT 'PUBLIC' FROM ME M WHERE M.IS_USER > 0 AND M.HAS_PUBLIC = 0),
WALK (NODE, DEPTH, PATH, SEEN) AS (   -- every simple path down; SEEN = the roles on it, so a cycle stops
    SELECT CAST(S.NODE AS VARCHAR), CAST(1 AS INTEGER),
           CAST(A.A_USER || ' > ' || S.NODE AS VARCHAR), CAST(CHR(31) || S.NODE || CHR(31) AS VARCHAR)
    FROM STARTS S CROSS JOIN ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.CHILD, W.DEPTH + 1, W.PATH || ' > ' || E.CHILD, W.SEEN || E.CHILD || CHR(31)
    FROM WALK W JOIN EDGE E ON E.PARENT = W.NODE CROSS JOIN ARGS A
    WHERE W.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.CHILD || CHR(31) IN W.SEEN) = 0),
ROLES AS (   -- one row per role reached: its shortest path, and how many paths lead to it
    SELECT W.NODE, W.DEPTH, W.PATH, COUNT(*) OVER (PARTITION BY W.NODE) AS ROLE_WAYS
    FROM WALK W
    QUALIFY ROW_NUMBER() OVER (PARTITION BY W.NODE ORDER BY W.DEPTH, W.PATH) = 1)
SELECT P.PRIVILEGE, P.GRANTED_ON, P.OBJ AS OBJECT_NAME, W.NODE AS VIA_ROLE, W.DEPTH, W.PATH,
       SUM(W.ROLE_WAYS) OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ) AS WAYS
FROM ROLES W JOIN PRIV P ON P.GRANTEE = W.NODE
QUALIFY ROW_NUMBER() OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ ORDER BY W.DEPTH, W.PATH) = 1
$$;

In [ ]:
CREATE OR REPLACE FUNCTION AQ_DB.AQ_SCHEMA.AQ_ROLE_ACCESS(ROLE_NAME VARCHAR, MAX_DEPTH NUMBER DEFAULT 50, SHOW_BUILTIN BOOLEAN DEFAULT FALSE)
RETURNS TABLE (PRIVILEGE VARCHAR, GRANTED_ON VARCHAR, OBJECT_NAME VARCHAR, VIA_ROLE VARCHAR, DEPTH NUMBER,
               PATH VARCHAR, WAYS NUMBER)
AS
$$
WITH RECURSIVE
ARGS AS (SELECT CAST(ROLE_NAME AS VARCHAR) AS A_ROLE,
                CAST(COALESCE(MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST(SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
PRIV AS (   -- every other grant is a privilege on an object (OWNERSHIP of a role too)
    SELECT G.GRANTEE, G.PRIVILEGE, G.GRANTED_ON, G.OBJ
    FROM GR G
    WHERE NOT (G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE')),
WALK (NODE, DEPTH, PATH, SEEN) AS (   -- every simple path down; SEEN = the roles on it, so a cycle stops
    SELECT CAST(A.A_ROLE AS VARCHAR), CAST(1 AS INTEGER), CAST(A.A_ROLE AS VARCHAR),
           CAST(CHR(31) || A.A_ROLE || CHR(31) AS VARCHAR)
    FROM ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.CHILD, W.DEPTH + 1, W.PATH || ' > ' || E.CHILD, W.SEEN || E.CHILD || CHR(31)
    FROM WALK W JOIN EDGE E ON E.PARENT = W.NODE CROSS JOIN ARGS A
    WHERE W.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.CHILD || CHR(31) IN W.SEEN) = 0),
ROLES AS (   -- one row per role reached: its shortest path, and how many paths lead to it
    SELECT W.NODE, W.DEPTH, W.PATH, COUNT(*) OVER (PARTITION BY W.NODE) AS ROLE_WAYS
    FROM WALK W
    QUALIFY ROW_NUMBER() OVER (PARTITION BY W.NODE ORDER BY W.DEPTH, W.PATH) = 1)
SELECT P.PRIVILEGE, P.GRANTED_ON, P.OBJ AS OBJECT_NAME, W.NODE AS VIA_ROLE, W.DEPTH, W.PATH,
       SUM(W.ROLE_WAYS) OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ) AS WAYS
FROM ROLES W JOIN PRIV P ON P.GRANTEE = W.NODE
QUALIFY ROW_NUMBER() OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ ORDER BY W.DEPTH, W.PATH) = 1
$$;

In [ ]:
CREATE OR REPLACE FUNCTION AQ_DB.AQ_SCHEMA.AQ_ROLE_HOLDERS(ROLE_NAME VARCHAR, MAX_DEPTH NUMBER DEFAULT 50, SHOW_BUILTIN BOOLEAN DEFAULT FALSE)
RETURNS TABLE (HOLDER VARCHAR, HOLDER_KIND VARCHAR, DEPTH NUMBER, PATH VARCHAR, WAYS NUMBER)
AS
$$
WITH RECURSIVE
ARGS AS (SELECT CAST(ROLE_NAME AS VARCHAR) AS A_ROLE,
                CAST(COALESCE(MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST(SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD,
           CASE WHEN G.GRANTED_TO = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS PARENT_KIND
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
MEMBER AS (   -- role -> user; PUBLIC -> every user
    SELECT U.ROLE_NAME AS NODE, U.USR FROM UR U WHERE U.ROLE_NAME <> 'PUBLIC'
    UNION ALL
    SELECT 'PUBLIC', X.USR FROM ALL_USERS X),
UP (NODE, KIND, DEPTH, PATH, SEEN) AS (   -- every simple path up; SEEN = the roles on it, so a cycle stops
    SELECT CAST(A.A_ROLE AS VARCHAR), CAST('ROLE' AS VARCHAR), CAST(1 AS INTEGER), CAST(A.A_ROLE AS VARCHAR),
           CAST(CHR(31) || A.A_ROLE || CHR(31) AS VARCHAR)
    FROM ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.PARENT, E.PARENT_KIND, U.DEPTH + 1, E.PARENT || ' > ' || U.PATH, U.SEEN || E.PARENT || CHR(31)
    FROM UP U JOIN EDGE E ON E.CHILD = U.NODE CROSS JOIN ARGS A
    WHERE U.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.PARENT || CHR(31) IN U.SEEN) = 0),
H AS (
    SELECT U.NODE AS HOLDER, U.KIND AS HOLDER_KIND, U.DEPTH, U.PATH FROM UP U WHERE U.DEPTH > 1
    UNION ALL
    SELECT M.USR, 'USER', U.DEPTH, M.USR || ' > ' || U.PATH FROM UP U JOIN MEMBER M ON M.NODE = U.NODE)
SELECT H.HOLDER, H.HOLDER_KIND, H.DEPTH, H.PATH, COUNT(*) OVER (PARTITION BY H.HOLDER, H.HOLDER_KIND) AS WAYS
FROM H
QUALIFY ROW_NUMBER() OVER (PARTITION BY H.HOLDER, H.HOLDER_KIND ORDER BY H.DEPTH, H.PATH) = 1
$$;

In [ ]:
CREATE OR REPLACE FUNCTION AQ_DB.AQ_SCHEMA.AQ_OBJECT_USERS(OBJECT_NAME VARCHAR, PRIVILEGE VARCHAR DEFAULT '', MAX_DEPTH NUMBER DEFAULT 50, SHOW_BUILTIN BOOLEAN DEFAULT FALSE)
RETURNS TABLE (GRANTEE VARCHAR, GRANTEE_KIND VARCHAR, PRIVILEGE VARCHAR, GRANTED_ON VARCHAR, HOLDER VARCHAR,
               DEPTH NUMBER, PATH VARCHAR, WAYS NUMBER)
AS
$$
WITH RECURSIVE
ARGS AS (SELECT CAST(OBJECT_NAME AS VARCHAR) AS A_OBJ, CAST(PRIVILEGE AS VARCHAR) AS A_PRIV,
                CAST(COALESCE(MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST(SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD,
           CASE WHEN G.GRANTED_TO = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS PARENT_KIND
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
PRIV AS (   -- every other grant is a privilege on an object (OWNERSHIP of a role too)
    SELECT G.GRANTEE, CASE WHEN G.GRANTED_TO = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS KIND,
           G.PRIVILEGE, G.GRANTED_ON, G.OBJ
    FROM GR G
    WHERE NOT (G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
MEMBER AS (   -- role -> user; PUBLIC -> every user
    SELECT U.ROLE_NAME AS NODE, U.USR FROM UR U WHERE U.ROLE_NAME <> 'PUBLIC'
    UNION ALL
    SELECT 'PUBLIC', X.USR FROM ALL_USERS X),
UP (PRIVILEGE, GRANTED_ON, HOLDER, NODE, KIND, DEPTH, PATH, SEEN) AS (   -- every simple path up from each grant
    SELECT CAST(P.PRIVILEGE AS VARCHAR), CAST(P.GRANTED_ON AS VARCHAR), CAST(P.GRANTEE AS VARCHAR),
           CAST(P.GRANTEE AS VARCHAR), CAST(P.KIND AS VARCHAR), CAST(1 AS INTEGER), CAST(P.GRANTEE AS VARCHAR),
           CAST(CHR(31) || P.GRANTEE || CHR(31) AS VARCHAR)
    FROM PRIV P CROSS JOIN ARGS A
    WHERE P.OBJ = A.A_OBJ AND (COALESCE(A.A_PRIV, '') = '' OR P.PRIVILEGE = A.A_PRIV) AND A.A_MAX >= 1
    UNION ALL
    SELECT U.PRIVILEGE, U.GRANTED_ON, U.HOLDER, E.PARENT, E.PARENT_KIND, U.DEPTH + 1, E.PARENT || ' > ' || U.PATH,
           U.SEEN || E.PARENT || CHR(31)
    FROM UP U JOIN EDGE E ON E.CHILD = U.NODE CROSS JOIN ARGS A
    WHERE U.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.PARENT || CHR(31) IN U.SEEN) = 0),
H AS (
    SELECT U.NODE AS GRANTEE, U.KIND AS GRANTEE_KIND, U.PRIVILEGE, U.GRANTED_ON, U.HOLDER, U.DEPTH, U.PATH FROM UP U
    UNION ALL
    SELECT M.USR, 'USER', U.PRIVILEGE, U.GRANTED_ON, U.HOLDER, U.DEPTH, M.USR || ' > ' || U.PATH
    FROM UP U JOIN MEMBER M ON M.NODE = U.NODE)
SELECT H.GRANTEE, H.GRANTEE_KIND, H.PRIVILEGE, H.GRANTED_ON, H.HOLDER, H.DEPTH, H.PATH,
       COUNT(*) OVER (PARTITION BY H.GRANTEE, H.GRANTEE_KIND, H.PRIVILEGE, H.GRANTED_ON) AS WAYS
FROM H
QUALIFY ROW_NUMBER() OVER (PARTITION BY H.GRANTEE, H.GRANTEE_KIND, H.PRIVILEGE, H.GRANTED_ON
                           ORDER BY H.DEPTH, H.PATH) = 1
$$;

### A. 1. Every role a user holds, direct and inherited

In [ ]:
SELECT * FROM TABLE(AQ_DB.AQ_SCHEMA.AQ_USER_ROLES($USER_NAME, $MAX_DEPTH)) ORDER BY DEPTH, ROLE_NAME;

### A. 2. Everything a user can finally do, and through which role

In [ ]:
SELECT * FROM TABLE(AQ_DB.AQ_SCHEMA.AQ_USER_ACCESS($USER_NAME, $MAX_DEPTH)) ORDER BY OBJECT_NAME, PRIVILEGE;

### A. 3. Does the user reach this object, and how? (the user's access, filtered)

In [ ]:
SELECT * FROM TABLE(AQ_DB.AQ_SCHEMA.AQ_USER_ACCESS($USER_NAME, $MAX_DEPTH))
WHERE OBJECT_NAME = $OBJECT_NAME ORDER BY PRIVILEGE;

### A. 4. Everything a role grants, its own and that of every role below it

In [ ]:
SELECT * FROM TABLE(AQ_DB.AQ_SCHEMA.AQ_ROLE_ACCESS($ROLE_NAME, $MAX_DEPTH)) ORDER BY OBJECT_NAME, PRIVILEGE;

### A. 5. Who holds a role, upwards: the roles above it and the users at the end

In [ ]:
SELECT * FROM TABLE(AQ_DB.AQ_SCHEMA.AQ_ROLE_HOLDERS($ROLE_NAME, $MAX_DEPTH)) ORDER BY HOLDER_KIND DESC, DEPTH, HOLDER;

### A. 6. Which users can use this object with this privilege (set PRIVILEGE to '' for any)

In [ ]:
SELECT * FROM TABLE(AQ_DB.AQ_SCHEMA.AQ_OBJECT_USERS($OBJECT_NAME, $PRIVILEGE, $MAX_DEPTH))
WHERE GRANTEE_KIND = 'USER' ORDER BY GRANTEE;

### A. 7. Users that reach the object through more than MIN_DEPTH roles

In [ ]:
SELECT * FROM TABLE(AQ_DB.AQ_SCHEMA.AQ_OBJECT_USERS($OBJECT_NAME, $PRIVILEGE, $MAX_DEPTH))
WHERE GRANTEE_KIND = 'USER' AND DEPTH > $MIN_DEPTH ORDER BY DEPTH DESC, GRANTEE;

### A. 8. A user's access summarized: objects per role and privilege

In [ ]:
SELECT VIA_ROLE, PRIVILEGE, COUNT(*) AS OBJECTS
FROM TABLE(AQ_DB.AQ_SCHEMA.AQ_USER_ACCESS($USER_NAME, $MAX_DEPTH))
GROUP BY VIA_ROLE, PRIVILEGE ORDER BY VIA_ROLE, PRIVILEGE;

## Path B: run the queries directly

Same questions, no functions. Each query is the content of its file in `access_queries/`; the inputs cell supplies
the `$NAME` values.

### B. 1. Every role a user holds, direct and inherited

In [ ]:
WITH RECURSIVE
ARGS AS (SELECT CAST($USER_NAME AS VARCHAR) AS A_USER,
                CAST(COALESCE($MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST($SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD,
           CASE WHEN G.GRANTED_ON = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS CHILD_KIND
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
ME AS (   -- joins, not IN: a NULL name would empty NOT IN
    SELECT (SELECT COUNT(*) FROM ALL_USERS X JOIN ARGS A ON X.USR = A.A_USER) AS IS_USER,
           (SELECT COUNT(*) FROM UR U JOIN ARGS A ON U.USR = A.A_USER WHERE U.ROLE_NAME = 'PUBLIC') AS HAS_PUBLIC),
STARTS AS (   -- the user's granted roles, plus PUBLIC when it is not granted
    SELECT U.ROLE_NAME AS NODE FROM UR U JOIN ARGS A ON U.USR = A.A_USER
    UNION ALL
    SELECT 'PUBLIC' FROM ME M WHERE M.IS_USER > 0 AND M.HAS_PUBLIC = 0),
WALK (NODE, KIND, DEPTH, PATH, SEEN) AS (   -- every simple path down; SEEN = the roles on it, so a cycle stops
    SELECT CAST(S.NODE AS VARCHAR), CAST('ROLE' AS VARCHAR), CAST(1 AS INTEGER),
           CAST(A.A_USER || ' > ' || S.NODE AS VARCHAR), CAST(CHR(31) || S.NODE || CHR(31) AS VARCHAR)
    FROM STARTS S CROSS JOIN ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.CHILD, E.CHILD_KIND, W.DEPTH + 1, W.PATH || ' > ' || E.CHILD, W.SEEN || E.CHILD || CHR(31)
    FROM WALK W JOIN EDGE E ON E.PARENT = W.NODE CROSS JOIN ARGS A
    WHERE W.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.CHILD || CHR(31) IN W.SEEN) = 0)
SELECT W.NODE AS ROLE_NAME, W.KIND AS ROLE_KIND, W.DEPTH, W.PATH,
       COUNT(*) OVER (PARTITION BY W.NODE) AS WAYS
FROM WALK W
QUALIFY ROW_NUMBER() OVER (PARTITION BY W.NODE ORDER BY W.DEPTH, W.PATH) = 1
ORDER BY DEPTH, ROLE_NAME;

### B. 2. Everything a user can finally do, and through which role

In [ ]:
WITH RECURSIVE
ARGS AS (SELECT CAST($USER_NAME AS VARCHAR) AS A_USER,
                CAST(COALESCE($MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST($SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
PRIV AS (   -- every other grant is a privilege on an object (OWNERSHIP of a role too)
    SELECT G.GRANTEE, G.PRIVILEGE, G.GRANTED_ON, G.OBJ
    FROM GR G
    WHERE NOT (G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
ME AS (   -- joins, not IN: a NULL name would empty NOT IN
    SELECT (SELECT COUNT(*) FROM ALL_USERS X JOIN ARGS A ON X.USR = A.A_USER) AS IS_USER,
           (SELECT COUNT(*) FROM UR U JOIN ARGS A ON U.USR = A.A_USER WHERE U.ROLE_NAME = 'PUBLIC') AS HAS_PUBLIC),
STARTS AS (   -- the user's granted roles, plus PUBLIC when it is not granted
    SELECT U.ROLE_NAME AS NODE FROM UR U JOIN ARGS A ON U.USR = A.A_USER
    UNION ALL
    SELECT 'PUBLIC' FROM ME M WHERE M.IS_USER > 0 AND M.HAS_PUBLIC = 0),
WALK (NODE, DEPTH, PATH, SEEN) AS (   -- every simple path down; SEEN = the roles on it, so a cycle stops
    SELECT CAST(S.NODE AS VARCHAR), CAST(1 AS INTEGER),
           CAST(A.A_USER || ' > ' || S.NODE AS VARCHAR), CAST(CHR(31) || S.NODE || CHR(31) AS VARCHAR)
    FROM STARTS S CROSS JOIN ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.CHILD, W.DEPTH + 1, W.PATH || ' > ' || E.CHILD, W.SEEN || E.CHILD || CHR(31)
    FROM WALK W JOIN EDGE E ON E.PARENT = W.NODE CROSS JOIN ARGS A
    WHERE W.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.CHILD || CHR(31) IN W.SEEN) = 0),
ROLES AS (   -- one row per role reached: its shortest path, and how many paths lead to it
    SELECT W.NODE, W.DEPTH, W.PATH, COUNT(*) OVER (PARTITION BY W.NODE) AS ROLE_WAYS
    FROM WALK W
    QUALIFY ROW_NUMBER() OVER (PARTITION BY W.NODE ORDER BY W.DEPTH, W.PATH) = 1)
SELECT P.PRIVILEGE, P.GRANTED_ON, P.OBJ AS OBJECT_NAME, W.NODE AS VIA_ROLE, W.DEPTH, W.PATH,
       SUM(W.ROLE_WAYS) OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ) AS WAYS
FROM ROLES W JOIN PRIV P ON P.GRANTEE = W.NODE
QUALIFY ROW_NUMBER() OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ ORDER BY W.DEPTH, W.PATH) = 1
ORDER BY OBJECT_NAME, PRIVILEGE;

### B. 3. Does the user reach this object, and how? (the user's access, filtered)

In [ ]:
SELECT *
FROM (
WITH RECURSIVE
ARGS AS (SELECT CAST($USER_NAME AS VARCHAR) AS A_USER,
                CAST(COALESCE($MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST($SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
PRIV AS (   -- every other grant is a privilege on an object (OWNERSHIP of a role too)
    SELECT G.GRANTEE, G.PRIVILEGE, G.GRANTED_ON, G.OBJ
    FROM GR G
    WHERE NOT (G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
ME AS (   -- joins, not IN: a NULL name would empty NOT IN
    SELECT (SELECT COUNT(*) FROM ALL_USERS X JOIN ARGS A ON X.USR = A.A_USER) AS IS_USER,
           (SELECT COUNT(*) FROM UR U JOIN ARGS A ON U.USR = A.A_USER WHERE U.ROLE_NAME = 'PUBLIC') AS HAS_PUBLIC),
STARTS AS (   -- the user's granted roles, plus PUBLIC when it is not granted
    SELECT U.ROLE_NAME AS NODE FROM UR U JOIN ARGS A ON U.USR = A.A_USER
    UNION ALL
    SELECT 'PUBLIC' FROM ME M WHERE M.IS_USER > 0 AND M.HAS_PUBLIC = 0),
WALK (NODE, DEPTH, PATH, SEEN) AS (   -- every simple path down; SEEN = the roles on it, so a cycle stops
    SELECT CAST(S.NODE AS VARCHAR), CAST(1 AS INTEGER),
           CAST(A.A_USER || ' > ' || S.NODE AS VARCHAR), CAST(CHR(31) || S.NODE || CHR(31) AS VARCHAR)
    FROM STARTS S CROSS JOIN ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.CHILD, W.DEPTH + 1, W.PATH || ' > ' || E.CHILD, W.SEEN || E.CHILD || CHR(31)
    FROM WALK W JOIN EDGE E ON E.PARENT = W.NODE CROSS JOIN ARGS A
    WHERE W.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.CHILD || CHR(31) IN W.SEEN) = 0),
ROLES AS (   -- one row per role reached: its shortest path, and how many paths lead to it
    SELECT W.NODE, W.DEPTH, W.PATH, COUNT(*) OVER (PARTITION BY W.NODE) AS ROLE_WAYS
    FROM WALK W
    QUALIFY ROW_NUMBER() OVER (PARTITION BY W.NODE ORDER BY W.DEPTH, W.PATH) = 1)
SELECT P.PRIVILEGE, P.GRANTED_ON, P.OBJ AS OBJECT_NAME, W.NODE AS VIA_ROLE, W.DEPTH, W.PATH,
       SUM(W.ROLE_WAYS) OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ) AS WAYS
FROM ROLES W JOIN PRIV P ON P.GRANTEE = W.NODE
QUALIFY ROW_NUMBER() OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ ORDER BY W.DEPTH, W.PATH) = 1
)
WHERE OBJECT_NAME = $OBJECT_NAME
ORDER BY PRIVILEGE;

### B. 4. Everything a role grants, its own and that of every role below it

In [ ]:
WITH RECURSIVE
ARGS AS (SELECT CAST($ROLE_NAME AS VARCHAR) AS A_ROLE,
                CAST(COALESCE($MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST($SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
PRIV AS (   -- every other grant is a privilege on an object (OWNERSHIP of a role too)
    SELECT G.GRANTEE, G.PRIVILEGE, G.GRANTED_ON, G.OBJ
    FROM GR G
    WHERE NOT (G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE')),
WALK (NODE, DEPTH, PATH, SEEN) AS (   -- every simple path down; SEEN = the roles on it, so a cycle stops
    SELECT CAST(A.A_ROLE AS VARCHAR), CAST(1 AS INTEGER), CAST(A.A_ROLE AS VARCHAR),
           CAST(CHR(31) || A.A_ROLE || CHR(31) AS VARCHAR)
    FROM ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.CHILD, W.DEPTH + 1, W.PATH || ' > ' || E.CHILD, W.SEEN || E.CHILD || CHR(31)
    FROM WALK W JOIN EDGE E ON E.PARENT = W.NODE CROSS JOIN ARGS A
    WHERE W.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.CHILD || CHR(31) IN W.SEEN) = 0),
ROLES AS (   -- one row per role reached: its shortest path, and how many paths lead to it
    SELECT W.NODE, W.DEPTH, W.PATH, COUNT(*) OVER (PARTITION BY W.NODE) AS ROLE_WAYS
    FROM WALK W
    QUALIFY ROW_NUMBER() OVER (PARTITION BY W.NODE ORDER BY W.DEPTH, W.PATH) = 1)
SELECT P.PRIVILEGE, P.GRANTED_ON, P.OBJ AS OBJECT_NAME, W.NODE AS VIA_ROLE, W.DEPTH, W.PATH,
       SUM(W.ROLE_WAYS) OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ) AS WAYS
FROM ROLES W JOIN PRIV P ON P.GRANTEE = W.NODE
QUALIFY ROW_NUMBER() OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ ORDER BY W.DEPTH, W.PATH) = 1
ORDER BY OBJECT_NAME, PRIVILEGE;

### B. 5. Who holds a role, upwards: the roles above it and the users at the end

In [ ]:
WITH RECURSIVE
ARGS AS (SELECT CAST($ROLE_NAME AS VARCHAR) AS A_ROLE,
                CAST(COALESCE($MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST($SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD,
           CASE WHEN G.GRANTED_TO = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS PARENT_KIND
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
MEMBER AS (   -- role -> user; PUBLIC -> every user
    SELECT U.ROLE_NAME AS NODE, U.USR FROM UR U WHERE U.ROLE_NAME <> 'PUBLIC'
    UNION ALL
    SELECT 'PUBLIC', X.USR FROM ALL_USERS X),
UP (NODE, KIND, DEPTH, PATH, SEEN) AS (   -- every simple path up; SEEN = the roles on it, so a cycle stops
    SELECT CAST(A.A_ROLE AS VARCHAR), CAST('ROLE' AS VARCHAR), CAST(1 AS INTEGER), CAST(A.A_ROLE AS VARCHAR),
           CAST(CHR(31) || A.A_ROLE || CHR(31) AS VARCHAR)
    FROM ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.PARENT, E.PARENT_KIND, U.DEPTH + 1, E.PARENT || ' > ' || U.PATH, U.SEEN || E.PARENT || CHR(31)
    FROM UP U JOIN EDGE E ON E.CHILD = U.NODE CROSS JOIN ARGS A
    WHERE U.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.PARENT || CHR(31) IN U.SEEN) = 0),
H AS (
    SELECT U.NODE AS HOLDER, U.KIND AS HOLDER_KIND, U.DEPTH, U.PATH FROM UP U WHERE U.DEPTH > 1
    UNION ALL
    SELECT M.USR, 'USER', U.DEPTH, M.USR || ' > ' || U.PATH FROM UP U JOIN MEMBER M ON M.NODE = U.NODE)
SELECT H.HOLDER, H.HOLDER_KIND, H.DEPTH, H.PATH, COUNT(*) OVER (PARTITION BY H.HOLDER, H.HOLDER_KIND) AS WAYS
FROM H
QUALIFY ROW_NUMBER() OVER (PARTITION BY H.HOLDER, H.HOLDER_KIND ORDER BY H.DEPTH, H.PATH) = 1
ORDER BY HOLDER_KIND DESC, DEPTH, HOLDER;

### B. 6. Which users can use this object with this privilege (set PRIVILEGE to '' for any)

In [ ]:
SELECT *
FROM (
WITH RECURSIVE
ARGS AS (SELECT CAST($OBJECT_NAME AS VARCHAR) AS A_OBJ, CAST($PRIVILEGE AS VARCHAR) AS A_PRIV,
                CAST(COALESCE($MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST($SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD,
           CASE WHEN G.GRANTED_TO = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS PARENT_KIND
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
PRIV AS (   -- every other grant is a privilege on an object (OWNERSHIP of a role too)
    SELECT G.GRANTEE, CASE WHEN G.GRANTED_TO = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS KIND,
           G.PRIVILEGE, G.GRANTED_ON, G.OBJ
    FROM GR G
    WHERE NOT (G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
MEMBER AS (   -- role -> user; PUBLIC -> every user
    SELECT U.ROLE_NAME AS NODE, U.USR FROM UR U WHERE U.ROLE_NAME <> 'PUBLIC'
    UNION ALL
    SELECT 'PUBLIC', X.USR FROM ALL_USERS X),
UP (PRIVILEGE, GRANTED_ON, HOLDER, NODE, KIND, DEPTH, PATH, SEEN) AS (   -- every simple path up from each grant
    SELECT CAST(P.PRIVILEGE AS VARCHAR), CAST(P.GRANTED_ON AS VARCHAR), CAST(P.GRANTEE AS VARCHAR),
           CAST(P.GRANTEE AS VARCHAR), CAST(P.KIND AS VARCHAR), CAST(1 AS INTEGER), CAST(P.GRANTEE AS VARCHAR),
           CAST(CHR(31) || P.GRANTEE || CHR(31) AS VARCHAR)
    FROM PRIV P CROSS JOIN ARGS A
    WHERE P.OBJ = A.A_OBJ AND (COALESCE(A.A_PRIV, '') = '' OR P.PRIVILEGE = A.A_PRIV) AND A.A_MAX >= 1
    UNION ALL
    SELECT U.PRIVILEGE, U.GRANTED_ON, U.HOLDER, E.PARENT, E.PARENT_KIND, U.DEPTH + 1, E.PARENT || ' > ' || U.PATH,
           U.SEEN || E.PARENT || CHR(31)
    FROM UP U JOIN EDGE E ON E.CHILD = U.NODE CROSS JOIN ARGS A
    WHERE U.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.PARENT || CHR(31) IN U.SEEN) = 0),
H AS (
    SELECT U.NODE AS GRANTEE, U.KIND AS GRANTEE_KIND, U.PRIVILEGE, U.GRANTED_ON, U.HOLDER, U.DEPTH, U.PATH FROM UP U
    UNION ALL
    SELECT M.USR, 'USER', U.PRIVILEGE, U.GRANTED_ON, U.HOLDER, U.DEPTH, M.USR || ' > ' || U.PATH
    FROM UP U JOIN MEMBER M ON M.NODE = U.NODE)
SELECT H.GRANTEE, H.GRANTEE_KIND, H.PRIVILEGE, H.GRANTED_ON, H.HOLDER, H.DEPTH, H.PATH,
       COUNT(*) OVER (PARTITION BY H.GRANTEE, H.GRANTEE_KIND, H.PRIVILEGE, H.GRANTED_ON) AS WAYS
FROM H
QUALIFY ROW_NUMBER() OVER (PARTITION BY H.GRANTEE, H.GRANTEE_KIND, H.PRIVILEGE, H.GRANTED_ON
                           ORDER BY H.DEPTH, H.PATH) = 1
)
WHERE GRANTEE_KIND = 'USER'
ORDER BY GRANTEE;

### B. 7. Users that reach the object through more than MIN_DEPTH roles

In [ ]:
SELECT *
FROM (
WITH RECURSIVE
ARGS AS (SELECT CAST($OBJECT_NAME AS VARCHAR) AS A_OBJ, CAST($PRIVILEGE AS VARCHAR) AS A_PRIV,
                CAST(COALESCE($MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST($SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD,
           CASE WHEN G.GRANTED_TO = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS PARENT_KIND
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
PRIV AS (   -- every other grant is a privilege on an object (OWNERSHIP of a role too)
    SELECT G.GRANTEE, CASE WHEN G.GRANTED_TO = 'DATABASE_ROLE' THEN 'DATABASE ROLE' ELSE 'ROLE' END AS KIND,
           G.PRIVILEGE, G.GRANTED_ON, G.OBJ
    FROM GR G
    WHERE NOT (G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
MEMBER AS (   -- role -> user; PUBLIC -> every user
    SELECT U.ROLE_NAME AS NODE, U.USR FROM UR U WHERE U.ROLE_NAME <> 'PUBLIC'
    UNION ALL
    SELECT 'PUBLIC', X.USR FROM ALL_USERS X),
UP (PRIVILEGE, GRANTED_ON, HOLDER, NODE, KIND, DEPTH, PATH, SEEN) AS (   -- every simple path up from each grant
    SELECT CAST(P.PRIVILEGE AS VARCHAR), CAST(P.GRANTED_ON AS VARCHAR), CAST(P.GRANTEE AS VARCHAR),
           CAST(P.GRANTEE AS VARCHAR), CAST(P.KIND AS VARCHAR), CAST(1 AS INTEGER), CAST(P.GRANTEE AS VARCHAR),
           CAST(CHR(31) || P.GRANTEE || CHR(31) AS VARCHAR)
    FROM PRIV P CROSS JOIN ARGS A
    WHERE P.OBJ = A.A_OBJ AND (COALESCE(A.A_PRIV, '') = '' OR P.PRIVILEGE = A.A_PRIV) AND A.A_MAX >= 1
    UNION ALL
    SELECT U.PRIVILEGE, U.GRANTED_ON, U.HOLDER, E.PARENT, E.PARENT_KIND, U.DEPTH + 1, E.PARENT || ' > ' || U.PATH,
           U.SEEN || E.PARENT || CHR(31)
    FROM UP U JOIN EDGE E ON E.CHILD = U.NODE CROSS JOIN ARGS A
    WHERE U.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.PARENT || CHR(31) IN U.SEEN) = 0),
H AS (
    SELECT U.NODE AS GRANTEE, U.KIND AS GRANTEE_KIND, U.PRIVILEGE, U.GRANTED_ON, U.HOLDER, U.DEPTH, U.PATH FROM UP U
    UNION ALL
    SELECT M.USR, 'USER', U.PRIVILEGE, U.GRANTED_ON, U.HOLDER, U.DEPTH, M.USR || ' > ' || U.PATH
    FROM UP U JOIN MEMBER M ON M.NODE = U.NODE)
SELECT H.GRANTEE, H.GRANTEE_KIND, H.PRIVILEGE, H.GRANTED_ON, H.HOLDER, H.DEPTH, H.PATH,
       COUNT(*) OVER (PARTITION BY H.GRANTEE, H.GRANTEE_KIND, H.PRIVILEGE, H.GRANTED_ON) AS WAYS
FROM H
QUALIFY ROW_NUMBER() OVER (PARTITION BY H.GRANTEE, H.GRANTEE_KIND, H.PRIVILEGE, H.GRANTED_ON
                           ORDER BY H.DEPTH, H.PATH) = 1
)
WHERE GRANTEE_KIND = 'USER' AND DEPTH > $MIN_DEPTH
ORDER BY DEPTH DESC, GRANTEE;

### B. 8. A user's access summarized: objects per role and privilege

In [ ]:
SELECT VIA_ROLE, PRIVILEGE, COUNT(*) AS OBJECTS
FROM (
WITH RECURSIVE
ARGS AS (SELECT CAST($USER_NAME AS VARCHAR) AS A_USER,
                CAST(COALESCE($MAX_DEPTH, 50) AS INTEGER) AS A_MAX, CAST($SHOW_BUILTIN AS BOOLEAN) AS A_BUILTIN),
G AS (   -- active grants to account and database roles; a database role is named DB.ROLE, as the app does
    SELECT CASE WHEN R.GRANTED_TO = 'DATABASE_ROLE'
                THEN CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME ELSE R.TABLE_CATALOG END || '.' || R.GRANTEE_NAME
                ELSE R.GRANTEE_NAME END AS GRANTEE,
           R.GRANTED_TO, R.PRIVILEGE, R.GRANTED_ON,
           CASE WHEN R.GRANTED_ON = 'DATABASE' THEN R.NAME   -- DATABASE rows may repeat the name in TABLE_CATALOG
                WHEN R.GRANTED_ON = 'SCHEMA' THEN R.TABLE_CATALOG || '.' || R.NAME
                ELSE COALESCE(R.TABLE_CATALOG || '.', '') || COALESCE(R.TABLE_SCHEMA || '.', '') || R.NAME END AS OBJ
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_ROLES R
    WHERE R.DELETED_ON IS NULL AND R.GRANTED_TO IN ('ROLE', 'DATABASE_ROLE')),
GR AS (   -- Snowflake's own grants (SNOWFLAKE.* database roles, system compute pools, ...) only with SHOW_BUILTIN
    SELECT G.* FROM G CROSS JOIN ARGS A
    WHERE A.A_BUILTIN
       OR NOT (LEFT(G.GRANTEE, 10) = 'SNOWFLAKE.' OR LEFT(G.GRANTEE, 7) = 'SYSTEM$'
               OR LEFT(G.GRANTEE, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.GRANTEE, 19) = 'SNOWFLAKE_LEARNING_'
               OR (G.GRANTEE = 'PUBLIC' AND (LEFT(G.OBJ, 10) = 'SNOWFLAKE.' OR LEFT(G.OBJ, 7) = 'SYSTEM$'
                   OR LEFT(G.OBJ, 20) = 'SYSTEM_COMPUTE_POOL_' OR LEFT(G.OBJ, 19) = 'SNOWFLAKE_LEARNING_')))),
EDGE AS (   -- role links; links granted to an admin role are left out: every custom role sits under SYSADMIN
    SELECT G.GRANTEE AS PARENT, G.OBJ AS CHILD
    FROM GR G
    WHERE G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE'
      AND G.GRANTEE NOT IN ('ACCOUNTADMIN', 'SECURITYADMIN', 'SYSADMIN', 'USERADMIN', 'ORGADMIN')),
PRIV AS (   -- every other grant is a privilege on an object (OWNERSHIP of a role too)
    SELECT G.GRANTEE, G.PRIVILEGE, G.GRANTED_ON, G.OBJ
    FROM GR G
    WHERE NOT (G.GRANTED_ON IN ('ROLE', 'DATABASE_ROLE') AND G.PRIVILEGE = 'USAGE')),
UR AS (
    SELECT U.GRANTEE_NAME AS USR, U.ROLE AS ROLE_NAME
    FROM SNOWFLAKE.ACCOUNT_USAGE.GRANTS_TO_USERS U WHERE U.DELETED_ON IS NULL),
ALL_USERS AS (   -- every user holds PUBLIC, granted or not
    SELECT U.USR FROM UR U
    UNION
    SELECT U.NAME FROM SNOWFLAKE.ACCOUNT_USAGE.USERS U WHERE U.DELETED_ON IS NULL),
ME AS (   -- joins, not IN: a NULL name would empty NOT IN
    SELECT (SELECT COUNT(*) FROM ALL_USERS X JOIN ARGS A ON X.USR = A.A_USER) AS IS_USER,
           (SELECT COUNT(*) FROM UR U JOIN ARGS A ON U.USR = A.A_USER WHERE U.ROLE_NAME = 'PUBLIC') AS HAS_PUBLIC),
STARTS AS (   -- the user's granted roles, plus PUBLIC when it is not granted
    SELECT U.ROLE_NAME AS NODE FROM UR U JOIN ARGS A ON U.USR = A.A_USER
    UNION ALL
    SELECT 'PUBLIC' FROM ME M WHERE M.IS_USER > 0 AND M.HAS_PUBLIC = 0),
WALK (NODE, DEPTH, PATH, SEEN) AS (   -- every simple path down; SEEN = the roles on it, so a cycle stops
    SELECT CAST(S.NODE AS VARCHAR), CAST(1 AS INTEGER),
           CAST(A.A_USER || ' > ' || S.NODE AS VARCHAR), CAST(CHR(31) || S.NODE || CHR(31) AS VARCHAR)
    FROM STARTS S CROSS JOIN ARGS A
    WHERE A.A_MAX >= 1
    UNION ALL
    SELECT E.CHILD, W.DEPTH + 1, W.PATH || ' > ' || E.CHILD, W.SEEN || E.CHILD || CHR(31)
    FROM WALK W JOIN EDGE E ON E.PARENT = W.NODE CROSS JOIN ARGS A
    WHERE W.DEPTH < A.A_MAX AND POSITION(CHR(31) || E.CHILD || CHR(31) IN W.SEEN) = 0),
ROLES AS (   -- one row per role reached: its shortest path, and how many paths lead to it
    SELECT W.NODE, W.DEPTH, W.PATH, COUNT(*) OVER (PARTITION BY W.NODE) AS ROLE_WAYS
    FROM WALK W
    QUALIFY ROW_NUMBER() OVER (PARTITION BY W.NODE ORDER BY W.DEPTH, W.PATH) = 1)
SELECT P.PRIVILEGE, P.GRANTED_ON, P.OBJ AS OBJECT_NAME, W.NODE AS VIA_ROLE, W.DEPTH, W.PATH,
       SUM(W.ROLE_WAYS) OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ) AS WAYS
FROM ROLES W JOIN PRIV P ON P.GRANTEE = W.NODE
QUALIFY ROW_NUMBER() OVER (PARTITION BY P.PRIVILEGE, P.GRANTED_ON, P.OBJ ORDER BY W.DEPTH, W.PATH) = 1
)
GROUP BY VIA_ROLE, PRIVILEGE ORDER BY VIA_ROLE, PRIVILEGE;